# Generic Single Point

<div class="alert alert-info">
<strong>Exercise: Subset a global dataset to run a generic single point case</strong><br><br>

1) Subset our global datasets to run for a single point

2) Create and run a case called **Boulder_bgc.generic** using the:
  - compset ``I1PtClm60BgcRsGs``, the 
  - resolution ``CLM_USRDAT``, and that 
  - applies ``USERMODS`` to use the subset data

In this tutorial we'll use the *subset_data* script to create a surface dataset and data atmosphere (datm) for a particular point.  Then we'll create a case using the CTSM workflow with the usermods created by *subset_data*.

</div>


## Background

### Subset_data

<div class="alert alert-info">
<strong>Exercise </strong><br>

- Activate ctsm_pylib and 
- Move into the tools/site_and_regional directory:

```
conda activate ctsm_pylib
cd /glade/u/home/$USER/code/my_clm_code/tools/site_and_regional
```

- Look at the arguments subset_data requires for single point applications:
```
./subset_data point --help
```

You should see lots of arguments, starting with:
```
  --lat PLAT            Single point latitude. [default: 42.5]
  --lon PLON            Single point longitude. [default: 287.8]
  --lon-type {180,360}  Whether longitudes are in the [-180, 180] format (centered around the
                        Prime/0th Meridian) or the [0, 360] format (centered around the 180th
                        Meridian). Choose by specifying the upper limit.
  --site SITE_NAME      Site name or tag. [default: ]
  --uniform-snowpack    Modify surface data to have a uniform snow fraction.
  --include-nonveg      Do not zero non-vegetation land units in the surface data.
  --cap-saturation      Modify surface data to not allow saturation excess.
  --evenly_split_cropland
                        Introduce equal areas of all crops
  --dompft [DOM_PFT ...]
                        Dominant PFT(s): if we set the grid to 100% one or multiple PFTs
                        [default: None]
  ....
```

The *subset_data* script provides lots of ways to customize your surface dataset! 
First let's create a place to store your data and usermods
```
mkdir ~/boulder_point_data
```


The example that follows highlights some of the common ways you can do this.  Specifically in this example we'll create a:
- surface dataset for a single point case centered around Boulder, Colorado that assumes 100% temperate needleleaf evergreen trees and uniform snow coverage across the gridcell.
- Data atmosphere subset from CRUJRA for the year 2020
- usermods that can point to the data we created.


```
./subset_data point --lat 40.015 --lon -105.27 \
                    --lon-type 180 --site Boulder \
                    --uniform-snowpack --dompft 1 \
                    --create-surface  \
                    --create-datm  \
                    --datm-syr 2020 --datm-eyr 2020 \
                    --create-user-mods \
                    --outdir ~/boulder_point_data \
                    --user-mods-dir ~/boulder_point_data/usermods                    
```

Look at the files created:

```
ls -1 ~/boulder_point_data
```

expected output includes:
```
datmdata
surfdata_Boulder_hist_2000_16pfts_c261014.nc
usermods
```

**Check Your Understanding**

What does your datm data look like?

What does your surface data look like?

*Hint: use `ncdump -h`*


</div>

### Usermods

A **usermod directory** (“user modifications” directory) is a bundle of case-specific settings that CLM/CIME can apply when you create or clone a case. It’s a convenient way to start from a known configuration rather than manually setting every option afterward. The usermods may include files such as CLM namelist settings (user_nl_clm), settings for other model components, and XML configuration changes. 

You pass the bundle to CIME when creating a case. For example:
```
./create_newcase ... --user-mods-dir ~/boulder_point_data/usermods
```

<div class="alert alert-info">
<strong>Exercise </strong><br>

You can look to see what usermods settings your Boulder cases will inherit. 
```
ls -1 ~/boulder_point_data/usermods
```

expected output includes:
```
shell_commands
user_nl_clm
user_nl_datm_streams
```

**Check Your Understanding**

How are Boulder site-level details passed to CLM? 

*Hint: look at files in the usermod directory*

</div>

## Get Started
Let's create a BGC case for a generic site in Boulder.


<div class="alert alert-info">
<strong>Exercise </strong><br>

- Move into the NEON usermods directory:

```
cd /glade/u/home/$USER/code/my_clm_code/cime_config/usermods_dirs/clm/NEON
```

- List its contents:
```
ls -1
```

You should see several directories, including:
```
ABBY
BARR
BART
BLAN
BONA
CLBJ
CPER
DCFS
defaults
DEJU
DELA
DSNY
FATES
GRSM
...
```

- You'll notice that FATES has its own directory.  Have a look inside.
```
ls FATES
```

- The `default` directory in NEON/FATES contains shell commands and user_nl_* settings that are shared across all sites. Look and see.
```
ls -1 FATES/defaults
```

expected output includes:
```
shell_commands
user_nl_clm
user_nl_cpl
user_nl_datm
user_nl_datm_streams
```
You can look to see what default settings your NEON cases will inherit. 

- Each NEON site also includes custom XML changes.  Look and see:
```
ls -1 FATES/BONA
```

expected output includes:
```
include_user_mods
shell_commands
```

**Check Your Understanding**

How are NEON site-level details passed to CLM? 

*Hint: look at the shell_commands in the CLM usermods_dirs for NEON*

</div>

## Get Started
Let's create a FATES sp case at a NEON site.


<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>
    
**Is my workspace set up?**

Did you already create a `cases` directory where you will put all new cases
```
cd /glade/u/home/$USER
mkdir cases
```
   
**Did you create a new case?**

You can create a new case with the command:
```
./create_newcase --case /glade/u/home/$USER/cases/CASE --res RES --compset COMPSET --user-mods-dir USERMODS --run-unsupported 
```
Make sure you use the correct information for the CASE, RES, COMPSET, and USERMODS inputs.


**How do you customize your case to run a FATES SP experiment?**

Specifically you'll want to make namelist changes that:
- set up a FATES-SP case.
- modify history file output for an SP case.
**NOTE** These should be done with a compset in CLM6, but we don't have a single point FATESsp compset at this time.

And XML changes to run:
- From a 'cold start' (as we don't have initial conditions);
- A quick spinup to initialize the soil moisture state (~4 years should be enough?);
- For the full observational record.

**Did you set up the case?**

You can invoke case set up as follows:
```
./case.setup
```
**How do I check my case**

You can check your CaseDocs (especially lnd_in) after you run:
```
./preview_namelists
```

**How do I compile?**

You can compile with the command:
```
qcmd -- ./case.build
```
**How do I check my job status?**

You can check job status using the command:
```
qstat -u $USER
```
    
</details>
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
                
### Create a new case <font face="Courier" color='purple'><strong>BONA.FATESsp </strong></font>:

```
cd /glade/u/home/$USER/code/my_clm_code/cime/scripts
```

Change the 4-character NEON site below.
```
export neon_site="BONA"
```

Then create a new case.
```
./create_newcase --case /glade/u/home/$USER/cases/${neon_site}.FATESsp \
    --res CLM_USRDAT \
    --compset I1PtClm60FatesRsGs \
    --run-unsupported \
    --user-mods-dir clm/NEON/FATES/${neon_site} \
    --handle-preexisting-dirs r
```

### Case <font face="Courier" color='purple'><strong>setup</strong></font>:

```    
cd /glade/u/home/$USER/cases/BONA.FATESsp
./case.setup
```

### Case <font face="Courier" color='purple'><strong>customization</strong></font>:
**user_nl_clm** changes.  We currently don't have a FATES sp compset in CLM.  We're working on it, but in the meantime you can request this by adding the following into user_nl_clm.
```    
echo "use_fates_sp = .true." >> user_nl_clm
echo "hist_fincl2 = 'FCEV','FCTR','FGEV','FIRA','FSA','FSH','FATES_GPP','FATES_GPP_PF','H2OSOI','SNOW_DEPTH','TBOT','TSOI'" >> user_nl_clm
```

**XML changes**
First look and see what default settings are for:
- Running from a cold start (without initial conditions)
- What years of DATM data do we have?
- How long are we running the case?
```
./xmlquery CLM_FORCE_COLDSTART
./xmlquery DATM_YR_START DATM_YR_ALIGN DATM_YR_END
./xmlquery STOP_OPTION STOP_N 
```

Expected results are:
```
	CLM_FORCE_COLDSTART: off

	DATM_YR_START: 2018
	DATM_YR_ALIGN: 2018
	DATM_YR_END: 2021

	STOP_OPTION: ndays
	STOP_N: 5

```

For most NEON sites we have at least 4 complete years of data to work with (2018-2021). 
Now we'll change this to:
- Force a cold start
- Cycle over the full 4 years of DATM data available two times 
- Start the simulation in 2014, then we'll have 4 years of spinup and 4 years to compare with the observed data.
```
./xmlchange CLM_FORCE_COLDSTART=on
./xmlchange STOP_OPTION=nyears
./xmlchange STOP_N=8  
./xmlchange RUN_STARTDATE=2014-01-01 
```

### Case <font face="Courier" color='purple'><strong>build</strong></font>:
```
qcmd -- ./case.build 
```


### <font face="Courier" color='purple'><strong>Preview namelist</strong></font>:

```./preview_namelists```

Check that your lnd_in file looks OK, e.g.:
- use_fates_sp = .true., 
- fsurdat = *16pfts*, 
- finidat = ' '
- etc...

```cat CaseDocs/lnd_in | head -50```

### Case <font face="Courier" color='purple'><strong>submit</strong></font>:
```
./case.submit
```

Check the status:
```
qstat -u $USER
```    

When the run is completed, look into the archive directory for: 
<font face="Courier" color='purple'><strong>BONA.FATESsp</strong></font>.  
    
Check that your archive directory on derecho (The path will be different on other machines): 
```
cd /glade/derecho/scratch/$USER/archive/BONA.FATESsp
ls 
```

</details>
</div>



## Test your understanding

- Did your run complete successfully?
- Where is your case directory?
- Where is your run directory?
- What sorts of files did you get as output?
- What years did you run over?